# Programming assignment 5: Dermatological image classification<a href="#Programming-assignment-5:-Dermatological-image-classification"
class="anchor-link">¶</a>

## To submit:<a href="#To-submit:" class="anchor-link">¶</a>

-   A Jupyter notebook (both in .ipynb and .html) containing the code of
    your solution as well as text cells describing what you have done.
    Make sure that the code is neat and the documentation readable.
-   A report that describes your work (as a PDF document).

Please **remember to join a Canvas group before submitting the
assignment**. You will need to repeat this procedure for every group
assignment. Also please **remember to include the names of the group
members** in the notebook and in the report.

## Background:<a href="#Background:" class="anchor-link">¶</a>

The tasks in this assignment are to:

-   classify images of skin lesions as **melanoma** (caused by a type of
    life-threatening skin cancer) or **nevus** (harmless)
-   write a technical report that describes your solutions and results.

Didactic purposes of this assignment:

-   working with ML architectures for images;
-   exploring and comparing a variety of solutions;
-   practising your technical writing skills.

References:

-   [Lindholm et al., 6.3 Convolutional neural
    networks](https://smlbook.org/book/sml-book-draft-latest.pdf#section.6.3)
-   Relevant lecture materials and example notebook(s).

### Introduction<a href="#Introduction" class="anchor-link">¶</a>

Skin cancer is one of the most common types of cancers, and it is
critical that dangerous cases are discovered early. In particular,
[melanoma](https://en.wikipedia.org/wiki/Melanoma) has a poor prognosis
if not treated before it
[metastasizes](https://en.wikipedia.org/wiki/Metastasis) (spreads to
other parts of the body). On the other hand, a full recovery is highly
probable if the melanoma is discovered and treated early: [the National
Cancer Institute](https://seer.cancer.gov/statfacts/html/melan.html)
estimates the 5-year survival rate to 99.6% for localized cases,
compared to 35% in case the cancer has spread.

Skin cancers such as melanoma can often be distinguished from harmless
birthmarks ([nevus](https://en.wikipedia.org/wiki/Nevus)) using visual
criteria. Melanomas tend to be larger than harmless birthmarks and often
have more irregular shapes and colors. Examples of nevus and melanoma
can be seen from
[here](https://challenge.isic-archive.com/landing/2017/44/).

### The ISIC challenges<a href="#The-ISIC-challenges" class="anchor-link">¶</a>

The fact that we often can distinguish malignant from harmless cases by
visual inspection makes it attractive to try to use image classification
techniques to make this process automatic, and for this reason there has
been plenty of research on ML solutions for this classification task.

The [ISIC challenges](https://challenge.isic-archive.com/) were a series
of workshops organized at machine learning conferences between 2016 and
2020. The were set up as competitions where participants could develop
technical solutions for the classification of skin lesions and compete
with other solutions.

The dataset we will be using in this assignment is a processed subset of
the [2018 ISIC challenge
dataset](https://challenge.isic-archive.com/landing/2019/). The 2018
competition is described in [this
paper](https://arxiv.org/abs/1902.03368) and it used a dataset
originally presented in [this
article](https://www.nature.com/articles/sdata2018161). The images were
obtained from hospitals in Austria and Australia and have been released
to the public domain. In the original ISIC task, participants had to
distinguish 9 types of skin marks. We have simplified this so that you
only have to distinguish two types: melanoma and [melanocytic
nevus](https://en.wikipedia.org/wiki/Melanocytic_nevus) (that is, a type
of harmless birthmark, or nevus for short).

### Preliminaries<a href="#Preliminaries" class="anchor-link">¶</a>

Download **a5_data_new.zip** and extract it in your working directory.

If you are using Colab, it is probably best to put the extracted
directory into your Google Drive storage. Since the file is rather
large, it may be impractical to download it in the notebook session
using wget. See
[here](https://www.cse.chalmers.se/~richajo/dat450/reading/using_colab.html)
for a description of how you can make a Drive folder reachable from your
Colab session.

When you extract this archive, you will see that there are two
directories: train (training data) and val (validation data). In the
training and validation directories, there are subdirectories
corresponding to the two categories: MEL and NV corresponding to
melanoma and nevus, respectively. The directories are structured in this
way to make it convenient for you to use an
[ImageFolder](https://pytorch.org/vision/main/generated/torchvision.datasets.ImageFolder.html),
which you can combine with a DataLoader to load images into PyTorch
tensors and to split the dataset into batches.

**Please note**: ImageFolder enumerates the classes in alphanumeric
order, which means that MEL will become class 0 and NV class 1. The
class subdirectories are also traversed in alphanumeric order, unless
you use a DataLoader where you have set shuffle=True.

The following piece of code uses ImageFolder and DataLoader. It selects
8 random images and displays them in your notebook using
[plt.imshow](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html).
You may need to adjust the directory to where you extracted your files.
The transpose is needed because the RGB dimension comes before the
height and width dimensions in PyTorch/Torchvision, while plt.imshow
requires the RGB dimension to go last.

    from torchvision.datasets import ImageFolder
    from torch.utils.data import DataLoader
    import matplotlib.pyplot as plt

    folder = ImageFolder('train', transform=torchvision.transforms.ToTensor())
    loader = DataLoader(folder, batch_size=8, shuffle=True)

    Xexamples, Yexamples = next(iter(loader))

    for i in range(8):
        plt.subplot(2,4,i+1)  
        img = Xexamples[i].numpy().transpose(1, 2, 0)    
        plt.imshow(img, interpolation='none')
        plt.title('NV' if Yexamples[i] else 'MEL')
        plt.xticks([])
        plt.yticks([])

Clarification: some students apparently experience technical problems
when displaying the images. This part is just to get a feel for the
dataset and it is not necessary to solve the assignment.

## Your tasks<a href="#Your-tasks" class="anchor-link">¶</a>

### Task1: Explore different solutions and evaluate on validation set<a
href="#Task1:-Explore-different-solutions-and-evaluate-on-validation-set"
class="anchor-link">¶</a>

Set up a machine learning architecture for an image classifier and train
it on the training set. Explore and compare different solutions and
evaluate them on the validation set. You can use accuracy score as the
performance metric to compare the different solutions.

To be eligible to receive a maximal score on this assignment, it is
necessary that you investigate the effect of applying the following
techniques:

-   some sort of normalization such as batch normalization, layer
    normalization, or group normalization,
-   residual connections,
-   data augmentation by applying [random
    transformations](https://pytorch.org/vision/stable/transforms.html)
    to the images,
-   transfer learning using a pre-trained model (see below for some
    hints).

**Clarification**: the intention here is that you should try to see
whether each of these methods gives an improvement over a baseline that
does not use it.

For a minimal pass, it is enough that you get some model that work
somewhat well. This dataset is balanced, so if your accuracy is around
0.5, then probably you have a problem.

You can use any method, except 1) systems that can already classify skin
marks, or 2) API-based services that can't be run locally. If you want
to switch from PyTorch to other frameworks such as JAX or TensorFlow, it
is OK but we won't be able to help you get unstuck if you mess up.

### Task2: Compute the accuracy score on test set<a href="#Task2:-Compute-the-accuracy-score-on-test-set"
class="anchor-link">¶</a>

You will receive the test set with the labels next week. Compute the
score on this dataset and include it in the report.

**Clarification**: Since you will receive your test set result quite
late, your discussions about your technical choices should be based on
the validation set results. You will have just one test set score, so
any comparisons between different solutions will have to be based on the
validation set.

### Task3: Write a technical report<a href="#Task3:-Write-a-technical-report" class="anchor-link">¶</a>

When you have investigated and compared your technical solutions, write
a report detailing your implementation, your experiments and analysis.

The submitted report should be at most 4 pages (excluding ethics,
limitations, and appendix). Use [the following
template](https://chalmers.instructure.com/courses/28899/files/3405885?wrap=1)
to write the report. It should be written as a conventional technical
report: a typical structure would be that you have sections for the
introduction, technical solutions, experiments/results, conclusion.

In addition to the standard technical sections mentioned above, you
should also include two sections called **Limitations** and **Ethical
considerations** respectively. This requirement is similar to what is
used at some technical conferences. In Limitations, you discuss about
issues that you think could be improved in your solution or that would
limit its real-world usefulness. In Ethical considerations, you should
discuss all ethical aspects that you think are relevant when developing
and applying your solution. These sections should be placed after the
conclusion and they can each be one or a couple of paragraphs. They do
not count towards the page limit of 4 pages.

You are allowed to include an appendix where you can mention low-level
technical details such as model layers, training hyperparameters. The
appendix does not count towards the page limit.

## Explainer: working with pre-trained models in PyTorch/Torchvision<a
href="#Explainer:-working-with-pre-trained-models-in-PyTorch/Torchvision"
class="anchor-link">¶</a>

This section gives a few hints about how to load pre-trained vision
models.

When applying these models, you have the option of
[fine-tuning](https://en.wikipedia.org/wiki/Fine-tuning_(deep_learning))
the pre-trained model or keeping it "frozen": that is, you do not update
the weights of the original model but just apply it as a feature
extractor. For the purposes of this assignment, we believe that using a
frozen model will be more convenient and efficient, but you might get a
better accuracy with fine-tuning.

The following code shows how to load the VGG-16 model. [This
paper](https://arxiv.org/abs/1409.1556) describes the model and some
other PyTorch-related information can be found
[here](https://pytorch.org/vision/main/models/generated/torchvision.models.vgg16.html).
This model has been trained on a part of the
[ImageNet](https://www.image-net.org/) dataset that includes 1,000
categories of various everyday objects. It is an early model and there
are more capable models now, but it should be applicable with most types
of hardware.

    weights_id = torchvision.models.VGG16_Weights.IMAGENET1K_V1
    vggmodel = torchvision.models.vgg16(weights=weights_id)
    vggmodel.eval()
    vggtransforms = weights_id.transforms()

Some technical comments about this code:

-   The first time you execute this code, the model weights will be
    downloaded from a repository. The weights will be cached if you run
    the code again.
-   The model can be initialized with different sets of weights. We use
    weights_id to identify which weights to use. In our case, we want to
    use the weights of the model that has been pre-trained on ImageNet.
-   We called .eval() to set components such as dropout and batch
    normalization to "evaluation mode" since we are not training the
    model now.
-   vggtransform keeps track of the image preprocessing that was used
    when this model was trained. You can use it as part of your
    transform pipeline for ImageFolder. For any pre-trained image model,
    it is crucial to use the appropriate preprocessing, because
    otherwise we may confuse the model when applying it.

If you print the model in a notebook cell, you will see a text
representation of the model structure similar to the following:

    VGG(
        (features): Sequential( ... lots of CNN layers here ...)
        (avgpool): AdaptiveAvgPool2d(output_size=(7, 7))
        (classifier): Sequential( ... MLP layers here ... )
    )

Conceptually, we may think of this as a pipeline consisting of a
"feature extractor" (convolutional part) followed by a fully connected
neural network classifier. If we want to apply the feature extractor in
a standalone fashion, we can call it as vggmodel.features(X) where X is
a batch of images. The output can then be used as the input for a
smaller model that we train for our specific task.

Hint: If you want to work with a frozen model as a feature extractor,
you can simply run it once on each of the datasets (the training,
validation, and test parts) and save the results to a file (for
instance, by calling
[np.save](https://numpy.org/doc/stable/reference/generated/numpy.save.html)),
which can then be re-loaded. Applying a large model is rather
time-consuming, so re-using the precomputed results will be much faster
than applying the model many times.

Hint: If you run into problems with memory here, it might be useful to
turn off gradient computation (since you will not be applying gradient
descent updates to the parameters of this model). To do this, you can
either call
[.detach()](https://pytorch.org/docs/stable/generated/torch.Tensor.detach.html)
on the output tensor to get rid of its computational graph, or call the
model in a torch.no_grad() block:

    with torch.no_grad():
        ... apply the model ...  

Note: If you use some other pre-trained model, its structure might be
different and you might need to call it in a different way.

**Clarification**: as mentioned above, you are allowed to use any
software or resource in this assignment except complete skin mark
classifiers and API-based solutions. So you can use any pre-trained
model, not only VGG16.

In \[ \]: